# Frequent Mental Distress Prediction Using Air Pollution

**SIADS 696 Final Project Report | Team 7**  
**Jaeah Kim, Kyle Rodriguez, Sophia Boettcher**

**Report format note.** This notebook is written as the final report artifact. All report figures are loaded from `outputs/visualizations/reproducible_report_figures/`, which is regenerated by `notebooks/06_visualization_generation/Final_Report_Reproducible_Visualizations.ipynb` from project data and model-output CSVs.

## Introduction

**Question: What problem are we trying to solve?**  
**Answer:** We ask whether county-level air quality, climate, and socioeconomic features can help predict future community mental-health burden. The outcome is CDC PLACES frequent mental distress (FMD): the percentage of adults reporting poor mental health for 14 or more days in the past month.

**Question: What impact would solving this problem have?**  
**Answer:** If county-level signals can forecast next-year FMD even modestly, public-health agencies could use the results as planning signals for outreach, prevention, and behavioral-health capacity. We do not claim individual diagnosis or causality; the practical value is better identification of where additional support may be needed.

**Question: What motivated this project?**  
**Answer:** Prior literature links environmental exposures, socioeconomic conditions, and mental health, but fewer projects test whether public county-level data can support leakage-safe prediction and exploratory profiling. The team was motivated by the possibility that publicly available environmental and demographic data could help identify future mental-health need while also showing where those data are insufficient.

**Question: What supervised and unsupervised methods did we use, and what is novel about our approach?**  
**Answer:** The supervised task predicts next-year FMD from current-year features using Dummy, ElasticNet, Random Forest, and Gradient Boosting regressors. The unsupervised task compares DBSCAN and K-Means clusters using non-FMD features, then evaluates FMD after clustering as a held-out outcome. Compared with related studies focused on individual cohorts or single exposure families, our project combines pollution, climate, ACS socioeconomic features, health-access proxies, industry composition, and county geography in a county-year predictive/exploratory pipeline.

**Question: What are the main findings?**  
**Answer:** The best supervised model was **Random Forest tuned**, with holdout RMSE **1.98** versus the Dummy baseline RMSE **2.43**. This suggests weak predictive signal, but holdout R² remains near zero, so the model is not policy-ready. The unsupervised analysis found one dominant county group plus smaller outlier groups. DBSCAN was useful for identifying unusual counties, while K-Means produced cleaner silhouette scores but still imbalanced clusters.

## Related Work

        **Question: What similar work exists, and how is our project different?**  
        **Answer:** The three most similar research lines study particulate matter/depression, neighborhood environment/mental health, and U.S. environmental exposure/mental-health outcomes. Our difference is that we build a U.S. county-year machine-learning panel and combine supervised next-year prediction with unsupervised county profiling.

        | Study / resource | 1-2 sentence summary | Difference from this project |
| --- | --- | --- |
| Park et al., PM2.5/PM10 and depression risk in KLoSA | This study found that prolonged particulate exposure was associated with elevated depression risk among middle-aged and older adults in South Korea. | It is individual/cohort-based and non-U.S.; our project is U.S. county-level and predictive/exploratory rather than causal. |
| Lei et al., neighborhood environment and mental health in China | This work links perceived pollution, neighborhood conditions, and socioeconomic status to mental health outcomes. | Our project uses measured AQI/climate variables and ML clustering instead of perception-based neighborhood measures. |
| Werder et al., Gulf Long-Term Follow-up Study | This U.S.-based study examines PM2.5, greenspace, and depression in the Southeastern United States. | Our project expands across U.S. counties and adds supervised next-year prediction plus unsupervised county profiling. |

## Data Sources

        **Question: Where are the data from, what formats did they use, what variables matter, how many records did we use, and what time period is covered?**  
        **Answer:** The final enriched panel contains **14,748 county-year rows**, **83 columns**, and **3,008 county/county-equivalent GEOIDs** from **2019-2023**. Data are merged by five-digit county GEOID. The panel contains 49 state/DC labels in the final enriched file. Daily AQI monitor-derived features are available for about **31.3%** of rows before imputation, while NOAA climate features cover about **100.0%** of rows.

        | Source | Format/location | Important variables | Coverage / role |
| --- | --- | --- | --- |
| CDC PLACES | data.cdc.gov CSV | FMD prevalence, lower/upper confidence intervals | Annual county-level mental-health outcome |
| EPA AirData annual | EPA CSV | Days with AQI, Max AQI, Median AQI, pollutant-defining days | Annual county air-quality predictors |
| EPA daily AQI | EPA ZIP/CSV | Daily AQI mean/std/p90/max, days above 100/150/200 | Daily exposure variability and extreme-event predictors; sparse monitor coverage |
| ACS 5-year tables | Census bulk/API files | Income, poverty, unemployment, education, insurance, industry composition | Socioeconomic and health-access controls |
| NOAA Climate at a Glance | JSON/CSV-derived panel | Annual temperature, precipitation, historical anomalies | County climate context |
| Census county boundaries | Shapefile ZIP | County geometries | Mapping and residual/cluster visualization |

## Feature Engineering

**Question: How did we get from raw data to final ML features?**  
**Answer:** We merged CDC PLACES, EPA AirData, ACS, NOAA, and Census geography by county GEOID; cleaned sentinel values; imputed missing numeric model inputs; engineered next-year FMD by shifting each county's FMD forward one year; standardized numeric features where needed; and excluded FMD/confidence intervals from clustering.

**Question: How did we handle noisy or missing data?**  
**Answer:** The EDA showed that daily AQI features are structurally missing in many counties because EPA monitors are sparse. We retained these features but imputed missing numeric values inside modeling pipelines and interpreted daily-AQI results cautiously. NOAA coverage was near-complete, while ACS and PLACES were harmonized to a common county-year panel.

### Integration of Jaeah's EDA Work

        **Question: How did Jaeah's exploratory work feed into the final project?**  
        **Answer:** Jaeah's EDA notebook informed the data-quality checks, rate engineering, environmental/health framing, and unsupervised county-profiling direction. Her notebook is included at `notebooks/jaeah_eda.ipynb`.

        | Jaeah EDA component | What it checked | How it is incorporated here |
| --- | --- | --- |
| Missingness audit | Examined AQI missingness and whether merged rows had systematic gaps. | The report explicitly discusses sparse daily AQI coverage and uses imputation rather than dropping monitor-sparse counties. |
| Geographic coverage checks | Checked state/geography representation and possible Puerto Rico/Hawaii/coverage issues. | The final pipeline reports county-year coverage, maps residuals, and treats geographic patterns as exploratory rather than causal. |
| Feature engineering rates | Converted raw poverty/unemployment counts into intensity-style measures to avoid county-size bias. | The final feature-engineering section emphasizes rates, standardized predictors, and leakage-safe target construction. |
| Univariate and bivariate EDA | Explored AQI distributions, socioeconomic skew, FMD outliers, environmental justice, and income/AQI/FMD relationships. | Those exploratory patterns motivate the combined pollution + socioeconomic + climate feature representation and the cautious weak-signal interpretation. |
| Spatial and temporal framing | Explored how patterns varied over time and across places. | The final report includes temporal next-year prediction, geographic residual mapping, and unsupervised county-profile visualizations. |

### Feature Families and Leakage Controls

        | Feature family | Examples | Role / leakage control |
| --- | --- | --- |
| Identifiers | geoid, county_name, state_name, year | Merge/split/reporting keys; not substantive predictors. |
| FMD outcome | mental_health_prevalence, lower_ci, upper_ci, next_year_fmd | Supervised target or held-out cluster outcome; excluded from unsupervised clustering. |
| Annual AQI | Max AQI, Median AQI, category days, pollutant days | Current-year exposure predictors. |
| Daily AQI | daily_aqi_std, daily_aqi_p90, threshold days | Exposure variability and extreme-event predictors with sparse monitor coverage. |
| ACS socioeconomic | median_income, population, poverty_count, unemployed_count | Community vulnerability controls. |
| Education/health/industry | pct_uninsured, pct_bachelors_or_higher, pct_manufacturing | Access, educational attainment, and industrial-trait predictors. |
| Climate | annual_avg_temp_f, annual_precip_inches, anomalies | Environmental context beyond AQI. |

## Part A. Supervised Learning

### Methods

**Question: What was the supervised workflow?**  
**Answer:** We framed supervised learning as regression: predict year *t+1* FMD from year *t* county-level features. The final holdout used 2022 features to predict 2023 FMD.

**Question: Which model families did we explore and why?**  
**Answer:** We used at least three diverse model families: a Dummy Regressor baseline, ElasticNet regularized linear regression, Random Forest bagged trees, and Gradient Boosting sequential trees. These cover baseline, linear, and nonlinear tree-ensemble mechanisms.

**Question: How did we tune models?**  
**Answer:** We used grouped cross-validation by feature year on the training period, then evaluated the chosen models on the 2022-to-2023 temporal holdout. Hyperparameter exploration included ElasticNet alpha/l1-ratio, Random Forest feature sampling and leaf size, and Gradient Boosting learning rate/depth/estimators.

### Supervised Evaluation: Overall Results

        **Question: Which metrics did we use and why?**  
        **Answer:** We report RMSE and MAE because FMD is continuous and measured in percentage points. RMSE penalizes large misses, MAE is easier to interpret as average absolute percentage-point error, and R² shows whether the model explains more variation than a mean baseline.

        **Question: How did the model families compare using CV mean and standard deviation?**  
        **Answer:** The Random Forest had the best holdout RMSE and the best CV RMSE mean, but the improvement over baseline was modest and uncertainty across year-based folds was large.

        | Model | Holdout RMSE | Holdout MAE | Holdout R2 | CV RMSE mean | CV RMSE SD | CV R2 mean | CV R2 SD |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Random Forest tuned | 1.98 | 1.65 | -0.02 | 2.02 | 0.63 | 0.01 | 0.50 |
| Gradient Boosting tuned | 2.30 | 1.91 | -0.37 | 2.16 | 0.49 | -0.09 | 0.44 |
| ElasticNet tuned | 2.31 | 1.97 | -0.38 | 2.32 | 0.59 | -0.27 | 0.55 |
| Dummy mean baseline | 2.43 | 1.95 | -0.54 | 2.73 | 0.46 | -0.70 | 0.50 |

![Figure 1. Supervised model comparison with CV mean and SD.](../../outputs/visualizations/reproducible_report_figures/figure_01_supervised_model_comparison.png)

![Figure 2. Predicted vs. actual FMD and residual distribution.](../../outputs/visualizations/reproducible_report_figures/figure_02_predictions_residuals.png)

![Figure 3. Geographic residual map from project Census boundaries and model predictions.](../../outputs/visualizations/reproducible_report_figures/figure_03_residual_map.png)

### Feature Importance and Ablation

        **Question: Which features contributed to the best supervised model?**  
        **Answer:** Random Forest feature importance ranked education and climate variables highly, including bachelor-degree share, temperature anomaly, lower educational attainment, high-school attainment, annual temperature, and median income. AQI features were present but did not dominate the model.

        **Question: What did the ablation analysis show?**  
        **Answer:** Removing education and health-access features increased RMSE the most. Some pollution/climate feature removals did not worsen holdout RMSE in this split, which supports the cautious conclusion that current county-level AQI variables are not strong standalone predictors of next-year FMD.

        | Removed feature family | Features removed | Holdout RMSE | RMSE delta |
| --- | --- | --- | --- |
| Education and health access | 5 | 2.06 | 0.08 |
| Industry composition | 13 | 2.01 | 0.03 |
| Annual AQI | 15 | 1.99 | 0.01 |
| None (full model) | 0 | 1.98 | 0.00 |
| Daily AQI exposure | 25 | 1.98 | -0.00 |
| Socioeconomic scale | 9 | 1.94 | -0.04 |
| Climate | 6 | 1.94 | -0.04 |

![Figure 4. Random Forest feature importances.](../../outputs/visualizations/reproducible_report_figures/figure_04_rf_feature_importance.png)

![Figure 5. Random Forest feature-family ablation.](../../outputs/visualizations/reproducible_report_figures/figure_05_rf_ablation.png)

### Sensitivity, Tradeoffs, and Failure Analysis

        **Question: How sensitive was the best model to hyperparameters?**  
        **Answer:** Random Forest performance was sensitive to leaf size and feature sampling. Smaller leaves with square-root feature sampling performed best; larger leaves were more stable but less accurate.

        | min_samples_leaf | max_features | Holdout RMSE | Holdout R2 |
| --- | --- | --- | --- |
| 1 | sqrt | 1.98 | -0.02 |
| 3 | sqrt | 2.00 | -0.04 |
| 8 | sqrt | 2.02 | -0.06 |
| 15 | sqrt | 2.05 | -0.09 |
| 1 | 0.7 | 2.06 | -0.10 |
| 1 | 0.5 | 2.06 | -0.10 |
| 1 | 1.0 | 2.06 | -0.11 |
| 3 | 0.7 | 2.07 | -0.12 |

![Figure 6. Random Forest hyperparameter sensitivity.](../../outputs/visualizations/reproducible_report_figures/figure_06_rf_sensitivity.png)

**Question: What tradeoffs did we identify?**  
        **Answer:** Random Forest offered the best accuracy but lower interpretability than ElasticNet. More flexible trees captured nonlinear structure but still compressed predictions toward the mean. The most important practical tradeoff is weak predictive improvement versus the risk of overinterpreting county-level associations.

        **Question: Where did prediction fail, and why might it have failed?**  
        **Answer:** The largest errors were mostly high-FMD counties that the model underpredicted. These failures likely reflect missing local variables such as behavioral-health infrastructure, rurality, disability burden, opioid-related distress, social isolation, and monitor coverage limitations.

        | County | State | Actual FMD | Prediction | Residual | Failure category |
| --- | --- | --- | --- | --- | --- |
| Roosevelt | Montana | 22.70 | 16.35 | 6.35 | High-distress tail underprediction |
| Big Horn | Montana | 22.90 | 16.74 | 6.16 | High-distress tail underprediction |
| Marshall | West Virginia | 23.00 | 16.96 | 6.04 | Appalachian high-FMD underprediction |
| Corson | South Dakota | 21.90 | 15.86 | 6.04 | High-distress tail underprediction |
| Hancock | West Virginia | 22.70 | 16.80 | 5.90 | Appalachian high-FMD underprediction |
| Ohio | West Virginia | 22.10 | 16.22 | 5.88 | Appalachian high-FMD underprediction |

**Question: What future improvements might address these failures?**  
**Answer:** Add provider density, rurality/urbanicity, opioid burden, disability prevalence, greenspace, disaster exposure, and finer monthly/daily exposure windows. The model needs richer local context to handle high-distress tails.

## Part B. Unsupervised Learning

        ### Methods

        **Question: What was the unsupervised workflow?**  
        **Answer:** We used the latest available county-year profile, standardized non-FMD features, reduced the feature space for visualization with PCA, clustered counties, and then compared FMD across clusters only after clustering.

        **Question: Which unsupervised methods did we use and why?**  
        **Answer:** We used DBSCAN and K-Means. DBSCAN is density-based and can flag outliers/noise, which is useful for identifying structurally unusual counties. K-Means is centroid-based and partitions every county, which gives a clean comparison method with a different clustering mechanism.

        **Question: How did we tune or explore unsupervised hyperparameters?**  
        **Answer:** DBSCAN was searched across `eps` and `min_samples`; K-Means was compared across values of `k`. We compared silhouette scores on the embedding and scaled features, cluster counts, noise rates, and interpretability.

        | Method | Selected parameters | Clusters | Noise % | Silhouette embedding | Silhouette scaled features |
| --- | --- | --- | --- | --- | --- |
| DBSCAN | eps=1.43, min_samples=5 | 2 | 0.022 | 0.76 | 0.45 |
| KMeans | k=2 | 2 | 0.000 | 0.89 | 0.75 |

### Unsupervised Evaluation

        **Question: Which unsupervised metrics did we use and why?**  
        **Answer:** We used silhouette scores to measure cluster separation, noise rate for DBSCAN, cluster counts/sizes for interpretability, and held-out FMD ANOVA to evaluate whether clusters formed without FMD were associated with mental-distress differences.

        | Method | F statistic | p-value | Groups |
| --- | --- | --- | --- |
| DBSCAN | 3.65 | 0.0563 | 2 |
| KMeans | 8.63 | 0.0033 | 2 |

**Question: What are the two visualizations for DBSCAN?**  
**Answer:** Figure 7 shows DBSCAN cluster separation on the PCA embedding. Figure 8 shows DBSCAN cluster profiles across average AQI, health access, education, temperature anomaly, and industry composition.

![Figure 7. DBSCAN clusters on PCA embedding.](../../outputs/visualizations/reproducible_report_figures/figure_07_dbscan_pca_scatter.png)

![Figure 8. DBSCAN cluster profiles.](../../outputs/visualizations/reproducible_report_figures/figure_08_dbscan_cluster_profile.png)

**Question: What are the two visualizations for K-Means?**  
**Answer:** Figure 9 shows K-Means cluster separation on the PCA embedding. Figure 10 shows K-Means cluster profiles using the same variables as DBSCAN, allowing direct comparison.

![Figure 9. K-Means clusters on PCA embedding.](../../outputs/visualizations/reproducible_report_figures/figure_09_kmeans_pca_scatter.png)

![Figure 10. K-Means cluster profiles.](../../outputs/visualizations/reproducible_report_figures/figure_10_kmeans_cluster_profile.png)

**Question: What was the unsupervised sensitivity analysis?**  
        **Answer:** We evaluated DBSCAN across multiple `eps`/`min_samples` settings and K-Means across values of `k`. K-Means `k=2` had the strongest silhouette score. DBSCAN showed high sensitivity to `eps`, with larger `eps` values producing fewer clusters and lower noise.

        | Method | Candidate parameters | Clusters | Noise % | Silhouette embedding | Silhouette scaled features |
| --- | --- | --- | --- | --- | --- |
| DBSCAN | eps=1.43, min_samples=5 | 2 | 0.022 | 0.76 | 0.45 |
| DBSCAN | eps=1.57, min_samples=5 | 3 | 0.018 | 0.74 | 0.43 |
| DBSCAN | eps=1.0, min_samples=5 | 2 | 0.033 | 0.65 | 0.28 |
| KMeans | k=2 | 2 | 0.000 | 0.89 | 0.75 |
| KMeans | k=3 | 3 | 0.000 | 0.70 | 0.35 |
| KMeans | k=6 | 6 | 0.000 | 0.21 | 0.08 |

![Figure 11. DBSCAN and K-Means unsupervised sensitivity search.](../../outputs/visualizations/reproducible_report_figures/figure_11_unsupervised_parameter_search.png)

## Discussion

**Question: What did we learn from Part A?**  
**Answer:** County-level environmental, climate, and socioeconomic data contain some predictive signal, but not enough for reliable next-year FMD forecasting. The best model modestly improved over baseline, but large errors remained in high-FMD counties.

**Question: What surprised us in Part A?**  
**Answer:** The weak holdout R² was surprising after enriching the dataset. Education, health access, and climate variables were more influential than expected, while AQI variables did not dominate prediction.

**Question: What challenges did we encounter in Part A, and how did we respond?**  
**Answer:** The main challenges were temporal leakage risk, sparse daily AQI coverage, and county-level aggregation. We responded by shifting the target forward within counties, using a temporal holdout, imputing sparse numeric features, and making cautious claims.

**Question: How would we extend Part A with more time?**  
**Answer:** Add healthcare provider density, rurality, opioid burden, disability prevalence, greenspace, disaster exposure, and monthly/daily exposure windows.

**Question: What did we learn from Part B?**  
**Answer:** Most counties occupy one broad socio-environmental space, while smaller groups/outliers differ structurally. DBSCAN is better for outlier detection, while K-Means is better for simple partitioning.

**Question: What surprised us in Part B?**  
**Answer:** The clusters were highly imbalanced, meaning the available county-level features did not form clean, evenly sized archetypes.

**Question: What challenges did we encounter in Part B, and how did we respond?**  
**Answer:** DBSCAN was sensitive to parameter choices and often produced one dominant cluster. We responded by comparing it to K-Means, reporting sensitivity results, and treating cluster interpretation as exploratory.

**Question: How would we extend Part B with more time?**  
**Answer:** Add richer social determinants, rurality/urbanicity classes, healthcare infrastructure, and possibly spatially constrained clustering or regional subgroup analyses.

## Ethical Considerations

**Question: What ethical issues arise for Part A supervised prediction?**  
**Answer:** County-level predictions could stigmatize communities or be misused as individual-level mental-health risk scores. They should be framed only as planning signals and paired with local expertise, uncertainty communication, and community input.

**Question: What ethical issues arise for Part B unsupervised clustering?**  
**Answer:** Cluster labels could be misread as rankings or causal categories. They should not be used to withdraw resources or blame communities. Clusters are exploratory summaries of observed county features, not definitive labels of community need or cause.

## Statement of Work

        **Question: What did each team member contribute?**  
        **Answer:** Each member contributed to data work, modeling, interpretation, and final synthesis, with the following primary responsibilities.

        | Team member | Primary contributions |
| --- | --- |
| Jaeah Kim | Exploratory data analysis, missingness and coverage checks, feature-engineering insight, unsupervised learning analysis support, clustering interpretation. |
| Kyle Rodriguez | Visualization support, supervised and unsupervised evaluation, repository cleanup, data gathering, model interpretation, report synthesis. |
| Sophia Boettcher | Dataset integration, imputation workflow, supervised and unsupervised baselines, external enrichment, final report drafting and rubric alignment. |
| All team members | EDA, data cleaning, modeling review, final interpretation, and presentation/report synthesis. |

## References

Centers for Disease Control and Prevention. (n.d.). *PLACES: Local Data for Better Health*. https://www.cdc.gov/places

Lei, K., Yang, J., & Ke, X. (2025). The impact of neighborhood environment on the mental health: Evidence from China. *Frontiers in Public Health, 12*, 1452744. https://doi.org/10.3389/fpubh.2024.1452744

National Oceanic and Atmospheric Administration. (n.d.). *Climate at a Glance*. National Centers for Environmental Information. https://www.ncei.noaa.gov/access/monitoring/climate-at-a-glance

Park, H., Kang, C., AiMS-CREATE Team, & Kim, H. (2024). Particulate matters (PM2.5, PM10) and the risk of depression among middle-aged and older population: Analysis of the Korean Longitudinal Study of Aging (KLoSA), 2016-2020 in South Korea. *Environmental Health, 23*(1), Article 4. https://doi.org/10.1186/s12940-023-01043-1

U.S. Census Bureau. (2009-2024). *American Community Survey 5-Year Estimates* [Data set]. Census Data API. https://api.census.gov/data

U.S. Census Bureau. (2023). *Cartographic Boundary Files: County Shapefile*. https://www.census.gov/geographies/mapping-files/time-series/geo/carto-boundary-file.html

U.S. Environmental Protection Agency. (2026). *Air Quality System (AQS) Data Mart: AirData* [Data set]. https://aqs.epa.gov/aqsweb/airdata/download_files.html

Werder, E., Lawrence, K., Deng, X., Jackson II, W. B., Christenbury, K., Buller, I., Engel, L., & Sandler, D. (2024). Residential air pollution, greenspace, and adverse mental health outcomes in the U.S. Gulf Long-term Follow-up Study. *Science of the Total Environment, 946*, 174434. https://doi.org/10.1016/j.scitotenv.2024.174434

## Project Submission Links

**Question: Where is the GitHub repository?**  
**Answer:** Insert the final GitHub URL here after the repository is pushed.

**Question: Where is the Google Doc version with comments enabled?**  
**Answer:** Insert the final Google Doc URL here after the report notebook/PDF and Google Doc are finalized.

## Appendix A. Complete Modeling Feature List

        **Question: What final features were used in supervised modeling?**  
        **Answer:** The following features appear in the final Random Forest feature-importance file and represent the final supervised modeling predictors used for the rubric-completion analysis.

        | Feature | Feature family | RF importance |
| --- | --- | --- |
| 90th Percentile AQI | Annual AQI | 0.0172 |
| Days CO | Annual AQI | 0.0032 |
| Days NO2 | Annual AQI | 0.0092 |
| Days Ozone | Annual AQI | 0.0119 |
| Days PM10 | Annual AQI | 0.0178 |
| Days PM2.5 | Annual AQI | 0.0112 |
| Days with AQI | Annual AQI | 0.0114 |
| Good Days | Annual AQI | 0.0122 |
| Hazardous Days | Annual AQI | 0.0009 |
| Max AQI | Annual AQI | 0.0222 |
| Median AQI | Annual AQI | 0.0163 |
| Moderate Days | Annual AQI | 0.0140 |
| Unhealthy Days | Annual AQI | 0.0074 |
| Unhealthy for Sensitive Groups Days | Annual AQI | 0.0199 |
| Very Unhealthy Days | Annual AQI | 0.0017 |
| annual_avg_temp_f | Climate | 0.0511 |
| annual_precip_anomaly_inches | Climate | 0.0242 |
| annual_precip_inches | Climate | 0.0296 |
| annual_temp_anomaly_f | Climate | 0.0722 |
| precip_1901_2000_mean | Climate | 0.0449 |
| temp_1901_2000_mean | Climate | 0.0289 |
| daily_aqi_category_days_good | Daily AQI exposure | 0.0032 |
| daily_aqi_category_days_hazardous | Daily AQI exposure | 0.0000 |
| daily_aqi_category_days_moderate | Daily AQI exposure | 0.0027 |
| daily_aqi_category_days_unhealthy | Daily AQI exposure | 0.0005 |
| daily_aqi_category_days_unhealthy_for_sensitive_groups | Daily AQI exposure | 0.0010 |
| daily_aqi_category_days_very_unhealthy | Daily AQI exposure | 0.0001 |
| daily_aqi_days_over_100 | Daily AQI exposure | 0.0012 |
| daily_aqi_days_over_150 | Daily AQI exposure | 0.0005 |
| daily_aqi_days_over_200 | Daily AQI exposure | 0.0001 |
| daily_aqi_defining_param_days_co | Daily AQI exposure | 0.0001 |
| daily_aqi_defining_param_days_no2 | Daily AQI exposure | 0.0007 |
| daily_aqi_defining_param_days_ozone | Daily AQI exposure | 0.0026 |
| daily_aqi_defining_param_days_pm10 | Daily AQI exposure | 0.0008 |
| daily_aqi_defining_param_days_pm25 | Daily AQI exposure | 0.0026 |
| daily_aqi_max | Daily AQI exposure | 0.0030 |
| daily_aqi_mean | Daily AQI exposure | 0.0035 |
| daily_aqi_median | Daily AQI exposure | 0.0023 |
| daily_aqi_monthly_mean_max | Daily AQI exposure | 0.0030 |
| daily_aqi_monthly_mean_std | Daily AQI exposure | 0.0036 |
| daily_aqi_months_observed | Daily AQI exposure | 0.0004 |
| daily_aqi_observed_days | Daily AQI exposure | 0.0024 |
| daily_aqi_p90 | Daily AQI exposure | 0.0026 |
| daily_aqi_reporting_sites_mean | Daily AQI exposure | 0.0015 |
| daily_aqi_std | Daily AQI exposure | 0.0034 |
| daily_aqi_unhealthy_or_worse_days | Daily AQI exposure | 0.0011 |
| pct_bachelors_or_higher | Education and health access | 0.0847 |
| pct_hs_or_higher | Education and health access | 0.0577 |
| pct_insured | Education and health access | 0.0147 |
| pct_less_than_hs | Education and health access | 0.0626 |
| pct_uninsured | Education and health access | 0.0151 |
| pct_ag_mining | Industry composition | 0.0125 |
| pct_arts_food | Industry composition | 0.0090 |
| pct_construction | Industry composition | 0.0071 |
| pct_education_health | Industry composition | 0.0093 |
| pct_finance_real_estate | Industry composition | 0.0190 |
| pct_information | Industry composition | 0.0091 |
| pct_manufacturing | Industry composition | 0.0114 |
| pct_other_services | Industry composition | 0.0073 |
| pct_professional_admin | Industry composition | 0.0219 |
| pct_public_admin | Industry composition | 0.0089 |
| pct_retail_trade | Industry composition | 0.0090 |
| pct_transport_utilities | Industry composition | 0.0081 |
| pct_wholesale_trade | Industry composition | 0.0092 |
| acs_civilian_noninstitutionalized_total | Socioeconomic scale | 0.0120 |
| acs_education_25plus_total | Socioeconomic scale | 0.0111 |
| acs_employed_total | Socioeconomic scale | 0.0147 |
| acs_uninsured_count | Socioeconomic scale | 0.0115 |
| median_income | Socioeconomic scale | 0.0478 |
| population | Socioeconomic scale | 0.0125 |
| poverty_count | Socioeconomic scale | 0.0175 |
| total_population_numeric | Socioeconomic scale | 0.0131 |
| unemployed_count | Socioeconomic scale | 0.0126 |

## Appendix B. Figure Provenance

**Question: Are report images generated from project data?**  
**Answer:** Yes. The figures in this notebook are loaded from `outputs/visualizations/reproducible_report_figures/`, and that folder is regenerated by `notebooks/06_visualization_generation/Final_Report_Reproducible_Visualizations.ipynb` using repository data and model-output CSVs. The figure manifest is `outputs/reproducible_report_figures/figure_manifest.csv`.